# Vector·Graph Hybrid RAG 시작하기

**Vector·Graph Hybrid RAG**는 의미가 가까운 문서를 먼저 찾고, 그 문서와 연결된 그래프 관계까지 따라가 답변 근거를 넓히는 RAG 방식이다.

문장 검색에는 Vector가 강하고, 대상 사이의 명시적 연결에는 Graph가 강하므로 두 장점을 순서대로 사용한다.

| 단계 | 입력 | 처리 | 출력 |
|---|---|---|---|
| Vector 검색 | 자연어 질문 | 질문과 `Document.text`의 임베딩을 비교한다. | 의미가 가까운 `Document` 후보와 `score` |
| Graph 확장 | 선택된 `Document` | `SUPPORTS` 관계를 Cypher로 따라간다. | 문서 본문과 연결된 `Course` |
| 답변 생성 | 확장된 검색 문맥 | LLM이 근거를 읽고 답한다. | 사용자용 답변 |

```mermaid
flowchart LR
    Q["자연어 질문"] --> E["질문 임베딩"]
    E --> V["Vector Index<br/>Document 후보 top_k"]
    V --> C["Cypher<br/>SUPPORTS 관계 확장"]
    C --> X["문서 본문 + Course + score"]
    X --> L["LLM 답변"]
```

Vector 검색이 후보와 순위를 정하고, Graph 조회는 각 후보에 관계 정보를 덧붙인다. <br>
이 실습은 Neo4j 안에 그래프와 Vector Index를 함께 두므로 Pinecone도 사용하지 않는다.

비정형 데이터는 `Document.text` 같은 문장이고, 구조화된 데이터는 `Document-[:SUPPORTS]->Course`처럼 방향과 의미가 정해진 관계이다.


[Neo4j 공식 VectorCypherRetriever 설명](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_rag.html#vector-cypher-retriever)

## 이름이 비슷한 Hybrid Retriever 구분하기

Retriever(리트리버)는 질문을 받아 LLM에 전달할 검색 항목을 반환하는 객체이다. 여기서 Vector search는 의미 유사도로, Full-text search는 단어·토큰 일치로 후보를 찾고, Cypher는 Neo4j의 관계 패턴을 따라간다.

| Retriever | 후보 검색 | Graph 관계 확장 | 이 노트북에서 사용 |
|---|---|---|---|
| `VectorCypherRetriever` | Vector | 수행 | 사용 |
| `HybridRetriever` | Vector + Full-text | 수행하지 않음 | 사용하지 않음 |
| `HybridCypherRetriever` | Vector + Full-text | 수행 | 사용하지 않음 |

이 노트북의 ‘Hybrid’는 두 검색 점수를 합친다는 뜻이 아니라 **Vector 후보 검색 뒤에 Graph 관계 확장을 연결한다**는 뜻이다. 따라서 실제 구현은 `VectorCypherRetriever`를 사용한다.

## 패키지와 실행 준비

`%pip`은 현재 Jupyter 커널의 Python 환경에 패키지를 설치하고, `-U`는 이미 설치된 패키지를 호환되는 최신 버전으로 갱신한다. 설치한 구성 요소는 역할별로 다음처럼 연결된다.

| 역할 | 구성 요소 | 이 노트북에서 하는 일 |
|---|---|---|
| 설정·연결 | `python-dotenv`, `GraphDatabase` | `.env`를 읽고 Neo4j driver를 만든다. |
| 벡터 준비 | `OpenAIEmbeddings`, `create_vector_index()`, `upsert_vectors()`, `EntityType.NODE` | 문서를 임베딩하고 노드에 벡터를 저장한다. |
| 검색 | `VectorCypherRetriever` | Vector 후보를 찾은 뒤 Cypher로 관계를 확장한다. |
| 생성 | `OpenAILLM`, `GraphRAG` | 검색 문맥을 사용자용 답변으로 바꾼다. |

`neo4j-graphrag[openai]`의 `[openai]`는 OpenAI 임베딩·LLM 연동에 필요한 선택 의존성을 함께 설치한다는 뜻이다. `text-embedding-3-small`의 기본 벡터 길이는 1536이므로 문서 벡터와 Vector Index의 `dimensions`도 1536으로 맞춘다.

공식 사용법은 [Neo4j GraphRAG VectorCypherRetriever](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_rag.html#vector-cypher-retriever)에서 확인할 수 있다.

In [1]:
%pip install -U neo4j-graphrag[openai] python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\playdata2\miniforge3\envs\llm_env\python.exe -m pip install --upgrade pip


## 환경 변수와 실행 설정 준비하기

`.env`는 API(Application Programming Interface) key와 DB(Database) 접속 정보를 코드 밖에 보관하는 파일이다. `load_dotenv()`가 값을 환경 변수로 올리고, 뒤의 연결·임베딩·생성 객체가 필요한 값을 읽는다. 실제 비밀값은 출력하지 않는다.

| 환경 변수 | 필수 여부 | 사용 위치 |
|---|---|---|
| `NEO4J_URI` | 필수 | Neo4j 서버 주소 |
| `NEO4J_USERNAME`, `NEO4J_PASSWORD` | 필수 | Neo4j 인증 |
| `OPENAI_API_KEY` | 필수 | 임베딩과 답변 생성 |
| `NEO4J_DATABASE` | 선택 | DB 이름, 기본값 `neo4j` |
| `OPENAI_CHAT_MODEL` | 선택 | 답변 모델, 기본값 `gpt-5.6-luna` |
| `OPENAI_EMBEDDING_MODEL` | 선택 | 임베딩 모델, 기본값 `text-embedding-3-small` |

필수 값은 `os.environ['이름']`으로 읽으므로 누락되면 해당 이름이 포함된 `KeyError`가 발생한다.

In [2]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.indexes import create_vector_index, upsert_vectors
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import EntityType

load_dotenv()

neo4j_database = os.getenv('NEO4J_DATABASE', 'neo4j')
chat_model = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.6-luna')
embedding_model = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')


## Neo4j 연결과 임베딩 모델 준비하기

Hybrid RAG는 서로 다른 일을 하는 세 객체를 연결한다.

| 객체 | 입력 → 출력 | 사용처 |
|---|---|---|
| `driver` | Cypher → Neo4j 조회 결과 | 노드·관계 저장과 검색 |
| `embedder` | 문장 → `list[float]` | 문서와 질문의 의미 비교 |
| `llm` | 검색 문맥 → 답변 문자열 | 최종 답변 생성 |

문서 저장과 질문 검색에는 같은 임베딩 모델을 사용해야 벡터 길이와 의미 공간이 일치한다. `driver`는 `Document-[:SUPPORTS]->Course` 관계를 조회하고, `llm`은 그 결과를 읽기 쉬운 답변으로 정리한다.

In [4]:
neo4j_uri = os.environ['NEO4J_URI']
neo4j_username = os.environ['NEO4J_USERNAME']
neo4j_password = os.environ['NEO4J_PASSWORD']
# OPENAI_API_KEY: 값은 출력하지 않고 존재 여부만 확인한다.
_ = os.environ['OPENAI_API_KEY']

driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password),
)

driver.verify_connectivity()

# 임베딩 객체
embedder = OpenAIEmbeddings(model=embedding_model)

llm = OpenAILLM(model_name=chat_model)

print('Neo4j·Embedding·LLM 객체 준비 완료')



Neo4j·Embedding·LLM 객체 준비 완료


## 문서 노드와 관계 적재하기

문서와 강좌를 다음 속성 그래프로 저장한다. 노드는 대상, 라벨은 대상의 종류, 속성은 대상의 세부 값, 관계는 두 대상의 연결 의미이다.

```mermaid
flowchart LR
    D["Document<br/>doc_id · text · embedding"] -->|SUPPORTS| C["Course<br/>title"]
```

`lecture_documents`의 한 딕셔너리가 `Document` 노드 하나와 연결할 `Course` 하나를 나타낸다. Cypher는 목록을 `$documents`로 받아 위 구조를 만들고, `elementId()`로 각 `Document`의 Neo4j 내부 식별자를 반환한다. 이 `element_id`는 다음 셀에서 올바른 노드에 임베딩을 저장할 때 사용한다.

이름이 비슷한 항목은 다음처럼 구분한다.

- `$documents`: Python의 `lecture_documents`가 들어가는 Cypher 매개변수이다.
- `document`: `UNWIND`가 목록에서 현재 꺼낸 딕셔너리 한 행이다. `document.doc_id`는 그 행의 `doc_id` 값이다.
- `doc`: 조회·생성한 Neo4j 문서 노드를 가리키는 Cypher 변수이다.
- `Document`: `doc` 노드의 종류를 나타내는 Label이다.
- `course`: 조회·생성한 강좌 노드를 가리키는 Cypher 변수이다.
- `Course`: `course` 노드의 종류를 나타내는 Label이다.
- `SUPPORTS`: `Document`에서 `Course`로 향하는 Relationship Type이다.

In [5]:
lecture_documents = [
    {
        'doc_id': 'doc-rag-01',
        'text': 'RAG는 외부 문서를 검색해 생성 모델의 근거 문맥으로 사용한다.',
        'course': 'RAG',
    },
    {
        'doc_id': 'doc-graph-01',
        'text': 'Graph RAG는 노드와 관계 경로를 조회해 답변 근거를 확장한다.',
        'course': 'Graph RAG',
    },
    {
        'doc_id': 'doc-mcp-01',
        'text': 'MCP는 모델과 외부 도구·데이터 소스의 연결 규약을 제공한다.',
        'course': 'MCP',
    },
]

write_query = '''
// 1. Python에서 받은 $documents 목록을 한 행씩 펼쳐 document라는 이름으로 참조한다.
UNWIND $documents AS document

// 2. doc_id가 같은 Document 노드를 찾고, 없으면 새로 만든다.
MERGE (doc:Document {doc_id: document.doc_id})
// 3. 새 노드와 기존 노드 모두 text 속성을 현재 입력 본문으로 갱신한다.
SET doc.text = document.text

// 4. title이 같은 Course 노드를 찾고, 없으면 새로 만든다.
MERGE (course:Course {title: document.course})
// 5. Document에서 Course로 향하는 SUPPORTS 관계를 찾거나 만든다.
MERGE (doc)-[:SUPPORTS]->(course)

// 6. 벡터 저장에 필요한 내부 ID와 문서 ID·본문을 결과 열로 반환한다.
RETURN elementId(doc) AS element_id, doc.doc_id AS doc_id, doc.text AS text
// 7. 다음 임베딩 목록의 순서가 일정하도록 doc_id를 기준으로 오름차순 정렬한다.
ORDER BY doc.doc_id
'''

document_records, write_summary, document_keys = driver.execute_query(

    write_query, # 실행할 Cypher 구문
    documents=lecture_documents, # $documents에 전달할 문서 목록
    database_=neo4j_database, # DB 이름
)

print('결과 열:', document_keys)
print('저장된 Document 수:', len(document_records))
print('새로 생성된 관계 수:', write_summary.counters.relationships_created)
print('첫 Document ID:', document_records[0]['doc_id'])

결과 열: ['element_id', 'doc_id', 'text']
저장된 Document 수: 3
새로 생성된 관계 수: 3
첫 Document ID: doc-graph-01


In [6]:
document_records

[<Record element_id='4:1d61d880-ed3f-44ee-aefd-b252a7a39bc0:11' doc_id='doc-graph-01' text='Graph RAG는 노드와 관계 경로를 조회해 답변 근거를 확장한다.'>,
 <Record element_id='4:1d61d880-ed3f-44ee-aefd-b252a7a39bc0:2' doc_id='doc-mcp-01' text='MCP는 모델과 외부 도구·데이터 소스의 연결 규약을 제공한다.'>,
 <Record element_id='4:1d61d880-ed3f-44ee-aefd-b252a7a39bc0:10' doc_id='doc-rag-01' text='RAG는 외부 문서를 검색해 생성 모델의 근거 문맥으로 사용한다.'>]

## 문서 임베딩과 Vector Index 저장하기

Embedding(임베딩)은 텍스트 의미를 실수 목록인 벡터로 바꾸는 변환이다. 이 단계는 **벡터 생성 → 노드 저장 → 검색 인덱스 구성**으로 나뉜다.

| 구성 요소 | 하는 일 | 하지 않는 일 |
|---|---|---|
| `embed_query(text)` | 문서 하나를 1536차원 벡터로 바꾼다. | Neo4j에 저장하지 않는다. |
| `upsert_vectors()` | `element_id`와 같은 위치의 벡터를 `Document.embedding`에 저장한다. | 후보를 검색하지 않는다. |
| `create_vector_index()` | 저장된 벡터를 빠르게 검색할 구조를 만든다. | 벡터 자체를 생성하지 않는다. |

`document_vectors`는 `document_records`와 순서가 같은 벡터 목록이다. Vector Index는 질문 벡터와 문서 벡터의 방향을 비교하는 `cosine` 유사도를 사용하며, 뒤의 Retriever가 이 값을 `score`로 반환한다.

In [7]:
# Neo4j에서 Document 임베딩 인덱스를 찾을 고정 이름
# == Vector Index 이름
VECTOR_INDEX_NAME = 'lecture_document_embeddings'

EMBEDDING_DIMENSIONS = 1536

# Neo4j Graph DB에 저장된 노드의 속성 중 text 속성만 얻어와
# 임베딩 수행하여 목록에 추가
document_vectors = [
    embedder.embed_query(record['text'])
    for record in document_records
]

# text를 벡터화 했을 때 차원 수가 1536가 아닌 경우
if any(len(vector) != EMBEDDING_DIMENSIONS
       for vector in document_vectors):
    raise ValueError("벡터의 차원 수가 지정된 차원 수와 일치하지 않음")




### Vector Index와 Document 임베딩 저장하기

차원 검사를 통과한 `document_vectors`를 Neo4j에 저장한다. `create_vector_index()`로 검색 구조를 먼저 준비하고, `upsert_vectors()`로 각 `element_id`의 `Document.embedding`에 벡터를 연결한다.

입력은 `document_records`의 element ID 목록과 같은 순서의 `document_vectors`이다. 출력으로 저장 개수·벡터 차원·인덱스 이름을 확인하고, 다음 Retriever가 `VECTOR_INDEX_NAME`으로 이 인덱스를 찾아 사용한다.

In [8]:
# 1. 벡터를 저장할 인덱스 생성
create_vector_index(
    driver = driver, # Neo4j 연결
    name = VECTOR_INDEX_NAME, # 생성할 INDEX 이름
    label='Document',  # embedding 속성을 가진 대상의 노드 라벨
    embedding_property='embedding', # 벡터를 저장하고 검색할 노드 속성명
    dimensions=EMBEDDING_DIMENSIONS, # 벡터 차원 수
    similarity_fn='cosine', # 유사도 비교 방식을 cosine 지정
    fail_if_exists=False, # 같은 이름의 인덱스가 있으면 오류X, 재사용O
    neo4j_database=neo4j_database # INDEX를 만들 대상 DB 이름
)

upsert_vectors(
    driver=driver,  # 벡터를 저장할 Neo4j 연결 객체
    ids=[record['element_id'] for record in document_records],  # Record 순서를 유지한 Document element ID 목록
    embedding_property='embedding',  # 각 Document에서 벡터를 저장할 속성명
    embeddings=document_vectors,
    entity_type=EntityType.NODE,  # 관계가 아니라 Document 노드에 벡터를 저장
    neo4j_database=neo4j_database,  # 벡터를 저장할 대상 DB 이름.
)

print('임베딩 저장 수:', len(document_vectors))
print('첫 임베딩 차원:', len(document_vectors[0]))
print('Vector Index:', VECTOR_INDEX_NAME)


임베딩 저장 수: 3
첫 임베딩 차원: 1536
Vector Index: lecture_document_embeddings


## Vector 후보에 Graph 관계 연결하기

`VectorCypherRetriever`는 하나의 Retriever 안에서 Vector 검색과 Graph 탐색을 순서대로 연결한다.

1. **입력**: 사용자의 `query_text` 문자열을 받는다.
2. **Vector 후보 선택**: 질문을 임베딩해 `lecture_document_embeddings`에서 가까운 `Document` 노드 `top_k`개를 찾는다.
3. **Graph 관계 확장**: 각 후보를 `node`, Vector 유사도를 `score`라는 이름으로 `retrieval_query`에 전달한다.
4. **context 구성**: `SUPPORTS` 관계를 따라 `Course`를 찾고 문서 본문·관련 강좌·Vector 유사도를 하나의 검색 항목으로 반환한다.

이 단계의 후보 순위와 `score`는 Vector 검색이 정한다. Graph 탐색은 별도 점수를 만들거나 순위를 합산하지 않고, 선택된 후보의 관계 정보를 보강한다.

`retrieval_query`를 정의하는 것만으로는 Neo4j를 조회하지 않는다. 뒤의 `vector_graph_retriever.search()`가 실행될 때 Retriever 내부의 Vector 검색 뒤에 붙어 각 후보마다 실행된다.

- `node`: Vector 검색이 선택해 넘긴 `Document` 후보이며, Python에서 선언한 변수가 아닌 Cypher 노드이다.
- `score`: 질문 벡터와 `node.embedding`의 유사도이며, Graph 탐색이 새로 계산한 값이 아닌 Vector 검색 결과이다.
- `course`: `node`와 연결된 강좌 노드를 가리키는 Cypher 변수이다.
- `Course`: `course`가 강좌 노드임을 제한하는 Label이다.
- `SUPPORTS`: `node`에서 `course`로 향하는 Relationship Type이다.
- `OPTIONAL MATCH`: 연결된 `Course`가 없어도 `Document` 후보 자체는 결과에 남긴다.
- `collect(course.title)`: 한 문서와 연결된 여러 강좌 이름을 하나의 목록으로 모은다.
- `AS doc_id`·`AS text`·`AS related_courses`: LLM 문맥에 들어갈 반환 열 이름을 정한다.

In [9]:
retrieval_query = '''
// 1. Vector 검색이 넘긴 Document 후보 node에서 SUPPORTS 관계를 따라 Course를 찾는다.
// OPTIONAL MATCH이므로 Course 관계가 없는 Document도 검색 결과에서 제거하지 않는다.
OPTIONAL MATCH (node)-[:SUPPORTS]->(course:Course)

// 2. Document 정보, 연결된 Course 목록, Vector 유사도 점수를 한 결과 행으로 반환한다.
RETURN node.doc_id AS doc_id,
       node.text AS text,
       // collect()는 한 Document에 연결된 여러 Course title을 list로 묶는다.
       collect(course.title) AS related_courses,
       // score는 앞선 Vector Index 검색이 계산해 retrieval_query로 전달한 값이다.
       score
'''

vector_graph_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME, # Document가 임베딩된 인덱스 이름

    # 검색된 Document(node)에서 Course 관계를 확장시키는 Cypher
    retrieval_query=retrieval_query,

    embedder=embedder, # 입력된 자연어 질문을 임베딩할 객체
    neo4j_database=neo4j_database,# 검색할 DB
)


### Vector·Graph 검색 결과 확인하기

`search()`는 `query_text`를 임베딩하고 Vector 유사도가 높은 문서를 최대 `top_k=2`개 선택한 뒤, 각 문서에서 `Course` 관계를 확장한다.

반환값은 `RetrieverResult`이며 실제 검색 항목은 `hybrid_result.items`에 들어 있다. 각 항목의 `content`에는 LLM에 전달할 문서·관계 문맥이 들어 있다. `metadata`는 부가 정보를 담을 수 있는 자리이지만 현재 결과에서는 `None`이고, 이 예제의 `score`는 `content` 안의 Record에 포함된다. 먼저 이 결과를 확인해야 검색 오류와 최종 생성 오류를 분리할 수 있다.

In [14]:

# hybrid_question = '외부 지식과 관계를 이용해 답변 근거를 확장하는 방법은?'
hybrid_question = '외부 도구와의 연결 규약은?'

hybrid_result = vector_graph_retriever.search(
    query_text=hybrid_question,
    top_k=2
)

print('검색 항목 수:', len(hybrid_result.items))
for index, item in enumerate(hybrid_result.items, start=1):
    print(f'[{index}] content:', item.content)
    print(f'[{index}] metadata:', item.metadata)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n// 1. Vector 검색이 넘긴 Document 후보 node에서 SUPPORTS 관계를 따라 Course를 찾는다.\n// OPTIONAL MATCH이므로 Course 관계가 없는 Document도 검색 결과에서 제거하지 않는다.\nOPTIONAL MATCH (node)-[:SUPPORTS]->(course:Course)\n\n// 2.

검색 항목 수: 2
[1] content: <Record doc_id='doc-mcp-01' text='MCP는 모델과 외부 도구·데이터 소스의 연결 규약을 제공한다.' related_courses=['MCP'] score=0.7917695045471191>
[1] metadata: None
[2] content: <Record doc_id='doc-rag-01' text='RAG는 외부 문서를 검색해 생성 모델의 근거 문맥으로 사용한다.' related_courses=['RAG'] score=0.6659429669380188>
[2] metadata: None


## 검색 문맥으로 답변 생성하기

앞 셀의 `hybrid_result`는 검색 결과이고 최종 답변이 아니다. Neo4j GraphRAG 패키지의 `GraphRAG` 클래스가 Retriever와 LLM을 연결해 생성 단계까지 수행한다.

**`hybrid_question` → `vector_graph_retriever.search()` → 검색 항목 → LLM용 context → `OpenAILLM` → `hybrid_answer.answer`**

`GraphRAG.search()`는 앞에서 출력한 `hybrid_result`를 직접 입력받지 않는다. 등록한 `vector_graph_retriever`를 다시 호출해 검색 문맥을 최종 프롬프트에 넣고 LLM 답변을 생성한다. 반환값인 `RagResultModel`은 최종 답변 문자열을 `answer` 속성에 담는다.

답변이 이상할 때는 어느 단계에서 문제가 생겼는지 나누어 확인한다.

| 점검 단계 | 확인 질문 | 문제가 있으면 볼 곳 |
|---|---|---|
| Vector 검색 | 의미가 가까운 `Document`가 후보로 선택되었는가? | 임베딩 모델, 질문, `top_k` |
| Graph 확장 | 후보와 연결된 `Course`가 반환되었는가? | 노드·관계 적재, `retrieval_query` |
| 답변 생성 | LLM이 context 안의 근거만 사용했는가? | 프롬프트, 검색 문맥 |

In [15]:
hybrid_rag = GraphRAG(
    retriever=vector_graph_retriever,
    llm=llm,
)

# hybrid_rag.search(자연어 질문)
# -> 자연어
# -> Vector DB -> Vector DB에서 유사도 높은 문서 검색
# -> Graph DB  -> 검색된 문서와 관계된 노드를 연결지어 반환
#      (== 최종 검색 결과 == LLM이 사용할 검색 근거)
# -> LLM 에게 질문 + 검색 결과를 전달
# -> 검색 결과를 근거로한 답변 반환
hybrid_answer = hybrid_rag.search(
    query_text=hybrid_question,
    retriever_config={'top_k':2}
)

print('최종 답변:', hybrid_answer)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n// 1. Vector 검색이 넘긴 Document 후보 node에서 SUPPORTS 관계를 따라 Course를 찾는다.\n// OPTIONAL MATCH이므로 Course 관계가 없는 Document도 검색 결과에서 제거하지 않는다.\nOPTIONAL MATCH (node)-[:SUPPORTS]->(course:Course)\n\n// 2.

최종 답변: answer='MCP입니다.' retriever_result=None


## 운영 선택 기준

검색 방식은 이름이 아니라 질문이 요구하는 근거에 맞춰 선택한다.

| 질문·데이터 조건 | 선택 |
|---|---|
| 문서의 의미 유사도만 필요하다. | Vector RAG |
| 자연어 질문을 관계 패턴으로 정확히 조회해야 한다. | Text2Cypher Graph RAG |
| 의미가 가까운 문서를 찾은 뒤 연결 엔터티가 필요하다. | `VectorCypherRetriever` |
| 의미 검색과 단어 검색을 함께 쓰되 관계 확장은 필요 없다. | `HybridRetriever` |
| 의미·단어 후보 결합 뒤 관계까지 확장해야 한다. | `HybridCypherRetriever` |

Hybrid는 인덱스와 관계를 함께 관리하므로 데이터 갱신·지연 시간·평가 비용이 증가한다. 단순 Vector RAG와 같은 질문 집합으로 비교 평가한 뒤 실제 이득이 있을 때 선택한다.

## Neo4j 연결 종료하기

`driver.close()`는 Python driver가 사용하는 연결 자원을 해제한다. Neo4j에 저장한 노드·관계·Vector Index는 삭제하지 않는다. 이 셀을 실행한 후 다시 검색하려면 driver를 새로 생성해야 한다.

In [16]:
driver.close()